# V2 Runtime Ready (Colab)
Notebook de prova E2E amb comprovacions fail-fast.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!test -d b-ia && (cd b-ia && git pull) || git clone https://github.com/rserrar/bia.git b-ia
%cd /content/b-ia

In [ ]:
!pip -q install requests numpy tensorflow

In [ ]:
import os

V2_API_BASE_URL = 'https://<api-real>'
V2_API_TOKEN = '<token-real>'
V2_API_PATH_PREFIX = ''

if '<api-real>' in V2_API_BASE_URL or '<token-real>' in V2_API_TOKEN:
    raise RuntimeError("Configura V2_API_BASE_URL i V2_API_TOKEN abans dexecutar.")

os.environ['PYTHONPATH'] = '/content/b-ia/V2'
os.environ['V2_API_BASE_URL'] = V2_API_BASE_URL
os.environ['V2_API_TOKEN'] = V2_API_TOKEN
os.environ['V2_API_PATH_PREFIX'] = V2_API_PATH_PREFIX
os.environ['V2_CODE_VERSION'] = 'colab-v2-runtime-ready'
os.environ['V2_CHECKPOINT_PATH'] = '/content/drive/MyDrive/bia_v2/run_state.json'
os.environ['V2_HEARTBEAT_INTERVAL_SECONDS'] = '30'
os.environ['V2_MAX_GENERATIONS'] = '2'
os.environ['V2_AUTO_PROCESS_PROPOSALS_PHASE0'] = 'true'
os.environ['V2_PROPOSALS_PHASE0_BATCH_SIZE'] = '20'

os.environ['V2_LLM_ENABLED'] = 'false'
os.environ['V2_LLM_USE_LEGACY_INTERFACE'] = 'true'
os.environ['V2_LLM_PROVIDER'] = 'openai'
os.environ['V2_LLM_ENDPOINT'] = 'https://api.openai.com/v1/chat/completions'
os.environ['V2_LLM_MODEL'] = 'gpt-5.4'
os.environ['V2_LLM_CONFIG_FILE'] = 'V2/config/llm_settings.json'
os.environ['V2_LLM_PROMPT_TEMPLATE_FILE'] = 'V2/prompts/generate_new_models.txt'
os.environ['V2_LLM_FIX_ERROR_PROMPT_FILE'] = 'V2/prompts/fix_model_error.txt'
os.environ['V2_LLM_ARCHITECTURE_GUIDE_FILE'] = 'V2/prompts/instruccions.md'
os.environ['V2_LLM_EXPERIMENT_CONFIG_FILE'] = 'V2/configs/experiment_config.json'

os.environ['V2_LEGACY_MODEL_JSON_PATH'] = 'V2/models/base/model_exemple_complex_v1.json'
os.environ['V2_LEGACY_EXPERIMENT_CONFIG_PATH'] = 'V2/configs/experiment_config.json'
os.environ['V2_LEGACY_BUILDER_PATH'] = 'V2/shared/utils/model_builder.py'

print('Entorn configurat correctament.')

In [ ]:
from pathlib import Path
import json

root = Path('/content/b-ia')
required_paths = [
    'V2/colab-worker/src/run_worker.py',
    'V2/colab-worker/src/engine.py',
    'V2/colab-worker/src/config.py',
    'V2/ops/scripts/run_phase0_model_validation.py',
    'V2/ops/scripts/run_generated_proposals_compile_check.py',
    'V2/ops/scripts/run_llm_full_prompt_check.py',
    'V2/ops/scripts/go_no_go_check.py',
    'V2/ops/scripts/probe_api_prefix.py',
    'V2/ops/configs/phase0_model_validation.json',
    'V2/configs/experiment_config.json',
    'V2/config/llm_settings.json',
    'V2/config/prompts_settings.json',
    'V2/prompts/generate_new_models.txt',
    'V2/prompts/fix_model_error.txt',
    'V2/prompts/instruccions.md',
    'V2/models/base/model_exemple_complex_v1.json',
    'V2/shared/utils/model_builder.py',
    'V2/shared/clients/api_client.py',
    'V2/shared/clients/llm_interface.py'
]
missing = [p for p in required_paths if not (root / p).exists()]
print(json.dumps({'ok': len(missing) == 0, 'missing': missing}, ensure_ascii=False, indent=2))
if missing:
    raise FileNotFoundError('Falten fitxers requerits a V2.')

In [ ]:
from pathlib import Path
import json
import shutil

root = Path('/content/b-ia')
cfg = json.loads((root / 'V2/configs/experiment_config.json').read_text(encoding='utf-8'))
data_base = (root / cfg['data_dir']).resolve()

compat_map = {
    'sortida_max.csv': 'sortida_max_7d.csv',
    'sortida_valors.csv': 'sortida_valors_7d.csv',
}
for src_name, dst_name in compat_map.items():
    src = data_base / src_name
    dst = data_base / dst_name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f'Compatibilitat: copiat {src_name} -> {dst_name}')

required_keys = [
    'entrada_valors','entrada_extra','min_historics','max_historics',
    'sortida_min_7d','sortida_max_7d','sortida_tb','sortida_sl','sortida_sn','sortida_valors_7d'
]
missing = []
for key in required_keys:
    filename = cfg['data_paths'][key]
    if not (data_base / filename).exists():
        missing.append({'key': key, 'file': filename})

print(json.dumps({'data_base': str(data_base), 'missing': missing}, ensure_ascii=False, indent=2))
if missing:
    raise FileNotFoundError('Dataset incomplet per al pipeline.')

In [ ]:
!python V2/ops/scripts/probe_api_prefix.py
!python V2/ops/scripts/go_no_go_check.py

In [ ]:
!python V2/ops/scripts/run_phase0_model_validation.py

In [ ]:
import os
os.environ['V2_LLM_ENABLED'] = 'true'
os.environ['V2_PROMPT_SEND_TO_LLM'] = 'false'
os.environ['V2_PROMPT_PUSH_TO_API'] = 'false'
os.environ['V2_PROMPT_REFERENCE_MODEL_PATH'] = 'V2/models/base/model_exemple_complex_v1.json'
!python V2/ops/scripts/run_llm_full_prompt_check.py

In [ ]:
import os
os.environ['V2_LLM_ENABLED'] = 'false'
os.environ['V2_MAX_GENERATIONS'] = '2'
!python V2/colab-worker/src/run_worker.py

In [ ]:
import os
import json
from pathlib import Path

checkpoint = Path(os.environ['V2_CHECKPOINT_PATH'])
if not checkpoint.exists():
    raise FileNotFoundError(f'Checkpoint no trobat: {checkpoint}')

state = json.loads(checkpoint.read_text(encoding='utf-8'))
run_id = state.get('run_id', '').strip()
if run_id == '':
    raise RuntimeError('No sha pogut recuperar run_id del checkpoint.')
os.environ['V2_TARGET_RUN_ID'] = run_id
print('run_id detectat:', run_id)

In [ ]:
!python V2/ops/scripts/run_generated_proposals_compile_check.py